# Train the original REleASE and HESPERIA matrices

This notebook now trains two deterministic forecasting paths:
- the original 2007 REleASE rise-parameter matrix
- the HESPERIA 30/60/90-minute ensemble matrix library

The original release path uses a 60-minute rise parameter and a 50x50 log-log lookup table.
The HESPERIA path trains multiple window-specific matrices for the three-matrix voter.

## Import Required Libraries

Load the libraries used for time-series preprocessing, binning, and JSON export.

In [25]:
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd

## Configure Training Parameters

Set the input columns, window sizes, bin counts, and output path for the trained matrix.

In [26]:
# Train on every historical CSV in gsep_ts, but skip the event list file.
GSEP_ROOT = Path("/Users/ran/cs/oires/data/raw/gsep_ts")
CSV_PATHS = sorted(
    str(path)
    for path in GSEP_ROOT.rglob("*.csv")
    if path.name.lower() != "gsep_list.csv"
)
TIME_COL = "time_tag"
ELECTRON_COL = "e2_flux_ic"
PROTON_COL = "p3_flux_ic"  # Target proton channel for training.
LEAD_MINUTES = 60
RELEASE_LOOKBACK_MINUTES = 60
HESPERIA_WINDOW_MINUTES = [30, 60, 90]
BIN_SIZES = [25, 35, 45]
MIN_SAMPLES_PER_CELL = 30
OUTPUT_DIR = Path("oneoutput")
RELEASE_MATRIX_JSON = OUTPUT_DIR / "trained_release_matrix.json"
print(f"Historical training files: {len(CSV_PATHS)}")
print(f"Excluded accuracy-check file: {GSEP_ROOT / 'GSEP_List.csv'}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"HESPERIA windows: {HESPERIA_WINDOW_MINUTES}")
print(f"Bin sizes: {BIN_SIZES}")
print(f"Original release matrix: {RELEASE_MATRIX_JSON}")

Historical training files: 433
Excluded accuracy-check file: /Users/ran/cs/oires/data/raw/gsep_ts/GSEP_List.csv
Output directory: oneoutput
HESPERIA windows: [30, 60, 90]
Bin sizes: [25, 35, 45]
Original release matrix: oneoutput/trained_release_matrix.json


## Helper Functions

Define the preprocessing, binning, and matrix-building helpers used by the trainer.

In [27]:
@dataclass
class TrainingRow:
    intensity: float
    delta: float
    target_proton: float


def read_series(csv_path: Path, time_col: str, electron_col: str, proton_col: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    for col in (time_col, electron_col, proton_col):
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in {csv_path}")

    out = df[[time_col, electron_col, proton_col]].copy()
    out[time_col] = pd.to_datetime(out[time_col], errors="coerce")
    out[electron_col] = pd.to_numeric(out[electron_col], errors="coerce")
    out[proton_col] = pd.to_numeric(out[proton_col], errors="coerce")
    out = out.dropna().sort_values(time_col).reset_index(drop=True)
    out.columns = ["time", "electron", "proton"]
    return out


def infer_cadence_seconds(times: pd.Series) -> float:
    if len(times) < 2:
        return 300.0
    diffs = times.diff().dropna().dt.total_seconds().to_numpy(dtype=float)
    if len(diffs) == 0:
        return 300.0
    return float(np.nanmedian(diffs))


def remove_background_gcr(values: np.ndarray, window_points: int) -> np.ndarray:
    window_points = max(1, window_points)
    s = pd.Series(values)
    background = s.rolling(window=window_points, min_periods=1).median()
    return (s - background).clip(lower=0.0).to_numpy(dtype=float)


def build_training_rows_for_file(
    df: pd.DataFrame,
    lead_minutes: int,
    delta_window_minutes: int,
) -> List[TrainingRow]:
    cadence_seconds = infer_cadence_seconds(df["time"])
    bg_window_points = max(3, int(round(3600.0 / max(cadence_seconds, 1.0))))
    cleaned_electron = remove_background_gcr(df["electron"].to_numpy(dtype=float), bg_window_points)

    delta_points = max(1, int(round((delta_window_minutes * 60.0) / max(cadence_seconds, 1.0))))
    lead_points = max(1, int(round((lead_minutes * 60.0) / max(cadence_seconds, 1.0))))

    proton = df["proton"].to_numpy(dtype=float)
    rows: List[TrainingRow] = []

    upper = len(df) - lead_points
    for i in range(delta_points, upper):
        intensity = float(cleaned_electron[i])
        delta = float(cleaned_electron[i] - cleaned_electron[i - delta_points])
        target = float(proton[i + lead_points])
        if np.isfinite(intensity) and np.isfinite(delta) and np.isfinite(target):
            rows.append(TrainingRow(intensity=intensity, delta=delta, target_proton=target))

    return rows


def safe_quantile_edges(values: np.ndarray, n_bins: int) -> np.ndarray:
    if len(values) == 0:
        raise ValueError("No values available for binning")

    qs = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.quantile(values, qs)
    edges = np.unique(edges)

    if len(edges) < 2:
        lo = float(values.min())
        hi = float(values.max())
        if hi <= lo:
            hi = lo + 1.0
        edges = np.linspace(lo, hi, n_bins + 1)

    if len(edges) - 1 < n_bins:
        lo = float(values.min())
        hi = float(values.max())
        if hi <= lo:
            hi = lo + 1.0
        edges = np.linspace(lo, hi, n_bins + 1)

    edges[0] = edges[0] - 1e-9
    edges[-1] = edges[-1] + 1e-9
    return edges


def cell_confidence(values: Sequence[float], min_samples_per_cell: int) -> float:
    n = len(values)
    if n == 0:
        return 0.0

    n_score = min(1.0, n / float(max(1, min_samples_per_cell)))
    vals = np.asarray(values, dtype=float)
    med = float(np.median(vals))
    q25 = float(np.percentile(vals, 25))
    q75 = float(np.percentile(vals, 75))
    iqr = q75 - q25
    scale = max(abs(med), 1.0)
    stability = max(0.0, 1.0 - min(1.0, iqr / (2.0 * scale)))
    return float(max(0.0, min(1.0, n_score * stability)))

## Build the Delta Matrix

Load the CSVs, generate training rows, bin the feature space, and compute the per-cell proton targets.

In [28]:
def build_delta_matrix(
    rows: Sequence[TrainingRow],
    intensity_bins: int,
    delta_bins: int,
    min_samples_per_cell: int,
) -> List[Dict[str, float]]:
    if not rows:
        raise ValueError("No training rows were produced. Check columns and data length.")

    intensity = np.array([r.intensity for r in rows], dtype=float)
    delta = np.array([r.delta for r in rows], dtype=float)
    target = np.array([r.target_proton for r in rows], dtype=float)

    i_edges = safe_quantile_edges(intensity, intensity_bins)
    d_edges = safe_quantile_edges(delta, delta_bins)

    i_idx = np.digitize(intensity, i_edges[1:-1], right=False)
    d_idx = np.digitize(delta, d_edges[1:-1], right=False)

    cell_values: Dict[Tuple[int, int], List[float]] = {}
    for ii, dd, tt in zip(i_idx, d_idx, target):
        key = (int(ii), int(dd))
        cell_values.setdefault(key, []).append(float(tt))

    global_median = float(np.median(target))
    matrix: List[Dict[str, float]] = []

    for ii in range(intensity_bins):
        for dd in range(delta_bins):
            vals = cell_values.get((ii, dd), [])
            target_peak = float(np.median(vals)) if vals else global_median
            conf = cell_confidence(vals, min_samples_per_cell=min_samples_per_cell)
            matrix.append(
                {
                    "intensity_min": float(i_edges[ii]),
                    "intensity_max": float(i_edges[ii + 1]),
                    "delta_min": float(d_edges[dd]),
                    "delta_max": float(d_edges[dd + 1]),
                    "target_peak_pfu": target_peak,
                    "p30_peak_pfu": target_peak,
                    "confidence": conf,
                }
            )

    return matrix

## Run Training Via CLI and Save JSON (Per Bin)

Call `train_delta_matrix.py` directly for each bin size in `BIN_SIZES`.

Each run writes to a bin-specific folder, for example `oneoutput/bin_25`, and stores a trained matrix JSON there.

In [30]:
import subprocess

RELEASE_TRAIN_SCRIPT = Path("/Users/ran/cs/6850/project/train_release_matrix.py")
HESPERIA_TRAIN_SCRIPT = Path("/Users/ran/cs/6850/project/train_delta_matrix.py")

release_train_command = [
    "python",
    str(RELEASE_TRAIN_SCRIPT),
    *CSV_PATHS,
    "--time-col",
    TIME_COL,
    "--electron-col",
    ELECTRON_COL,
    "--proton-col",
    PROTON_COL,
    "--lead-minutes",
    str(LEAD_MINUTES),
    "--lookback-minutes",
    str(RELEASE_LOOKBACK_MINUTES),
    "--intensity-bins",
    "50",
    "--rise-bins",
    "50",
    "--min-samples-per-cell",
    str(MIN_SAMPLES_PER_CELL),
    "--out-json",
    str(RELEASE_MATRIX_JSON),
]

print("Running:")
print(" ".join(release_train_command))
completed_release_train = subprocess.run(release_train_command, capture_output=True, text=True, check=False)
print("Return code:", completed_release_train.returncode)
print(completed_release_train.stdout)
print(completed_release_train.stderr)
if completed_release_train.returncode != 0:
    raise RuntimeError("train_release_matrix.py failed; see output above")

if not RELEASE_MATRIX_JSON.exists():
    raise RuntimeError(f"Expected release matrix not found: {RELEASE_MATRIX_JSON}")

print(f"Wrote original REleASE matrix: {RELEASE_MATRIX_JSON}")

TRAINED_BIN_CONFIGS = []
for bin_size in BIN_SIZES:
    bin_dir = OUTPUT_DIR / f"bin_{bin_size}"
    bin_dir.mkdir(parents=True, exist_ok=True)

    train_command = [
        "python",
        str(HESPERIA_TRAIN_SCRIPT),
        *CSV_PATHS,
        "--time-col",
        TIME_COL,
        "--electron-col",
        ELECTRON_COL,
        "--proton-col",
        PROTON_COL,
        "--instrument",
        "GOES",
        "--lead-minutes",
        str(LEAD_MINUTES),
        "--sliding-window-minutes",
        *[str(window) for window in HESPERIA_WINDOW_MINUTES],
        "--intensity-bins",
        str(bin_size),
        "--delta-bins",
        str(bin_size),
        "--min-samples-per-cell",
        str(MIN_SAMPLES_PER_CELL),
        "--out-dir",
        str(bin_dir),
    ]

    print("Running:")
    print(" ".join(train_command))
    completed_train = subprocess.run(train_command, capture_output=True, text=True, check=False)
    print("Return code:", completed_train.returncode)
    print(completed_train.stdout)
    print(completed_train.stderr)

    if completed_train.returncode != 0:
        raise RuntimeError(f"train_delta_matrix.py failed for bin {bin_size}; see output above")

    matrix_library_json = bin_dir / "matrix_index_goes.json"
    if not matrix_library_json.exists():
        raise RuntimeError(f"Expected HESPERIA matrix library not found: {matrix_library_json}")

    TRAINED_BIN_CONFIGS.append(
        {
            "bin_size": int(bin_size),
            "bin_dir": str(bin_dir),
            "matrix_library_json": str(matrix_library_json.resolve()),
            "release_matrix_json": str(RELEASE_MATRIX_JSON.resolve()),
        }
    )

print(f"Completed training for {len(TRAINED_BIN_CONFIGS)} bin settings")

Running:
python /Users/ran/cs/6850/project/train_release_matrix.py /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1986-02-03_21-25.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1986-02-05_05-50.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1986-02-05_23-00.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1986-02-07_01-00.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1986-02-10_09-00.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1986-02-14_02-20.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1986-02-16_14-20.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1986-03-06_06-25.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1986-05-04_01-00.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1987-05-29_05-25.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1987-11-07_19-50.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1987-12-29_14-15.csv /Users/ran/cs/oires/data/raw/gsep_ts/gsep_sc22_ts/1988-01-02_15-55.csv /Users/ran

## Generate Matrix Graphs

Run the standalone grapher script to create target and confidence heatmaps from the trained matrix JSON.

In [ ]:
import subprocess

GRAPHER_SCRIPT = Path("/Users/ran/cs/6850/project/delta_matrix_grapher.py")

# original RELeASE

bin_size = int(50)
matrix_json = Path(cfg["release_matrix_json"])
target_heatmap_png = bin_dir / f"matrix_target_heatmap_bin_{bin_size}.png"
conf_heatmap_png = bin_dir / f"matrix_confidence_heatmap_bin_{bin_size}.png"

graph_command = [
    "python",
    str(GRAPHER_SCRIPT),
    "--matrix-json",
    str(matrix_json),
    "--out-target-png",
    str(target_heatmap_png),
    "--out-confidence-png",
    str(conf_heatmap_png),
    "--target-field",
    "auto",
    "--title-prefix",
    f"REleASE Delta Matrix (bin={bin_size})",
]

print("Running:")
print(" ".join(graph_command))
completed_graph = subprocess.run(graph_command, capture_output=True, text=True, check=False)
print("Return code:", completed_graph.returncode)
print(completed_graph.stdout)
print(completed_graph.stderr)

# HESPERIA RELeAse
for cfg in TRAINED_BIN_CONFIGS:
    bin_size = int(cfg["bin_size"])
    bin_dir = Path(bin_dir)
    matrix_json = Path(cfg["release_matrix_json"])
    target_heatmap_png = bin_dir / f"matrix_target_heatmap_bin_{bin_size}.png"
    conf_heatmap_png = bin_dir / f"matrix_confidence_heatmap_bin_{bin_size}.png"

    graph_command = [
        "python",
        str(GRAPHER_SCRIPT),
        "--matrix-json",
        str(matrix_json),
        "--out-target-png",
        str(target_heatmap_png),
        "--out-confidence-png",
        str(conf_heatmap_png),
        "--target-field",
        "auto",
        "--title-prefix",
        f"REleASE Delta Matrix (bin={bin_size})",
    ]

    print("Running:")
    print(" ".join(graph_command))
    completed_graph = subprocess.run(graph_command, capture_output=True, text=True, check=False)
    print("Return code:", completed_graph.returncode)
    print(completed_graph.stdout)
    print(completed_graph.stderr)

## Run RELease With Trained Matrices (Per Bin)

Run RELease for every CSV path and every bin size.

Each bin setting writes to its own output folder, then runs hit/miss comparison to enable side-by-side evaluation.

In [ ]:
RELEASE_SCRIPT = Path("/Users/ran/cs/6850/project/RELease.py")
COMPARE_SCRIPT = Path("/Users/ran/cs/6850/project/compare_hit_miss.py")

RELEASE_INPUT_PATHS = list(CSV_PATHS)
RELEASE_INSTRUMENT = "GOES"
GSEP_LIST_CSV = GSEP_ROOT / "GSEP_List.csv"
ORIGINAL_RELEASE_OUTPUT_DIR = OUTPUT_DIR / "original_release_outputs"

print(f"Release script: {RELEASE_SCRIPT}")
print(f"Compare script: {COMPARE_SCRIPT}")
print(f"Total RELease inputs: {len(RELEASE_INPUT_PATHS)}")
print(f"Original release output dir: {ORIGINAL_RELEASE_OUTPUT_DIR}")
print(f"HESPERIA run configs: {len(TRAINED_BIN_CONFIGS)}")
for cfg in TRAINED_BIN_CONFIGS:
    print(
        f" - bin={cfg['bin_size']}: matrix_library={cfg['matrix_library_json']} -> output_dir={cfg['bin_dir']}"
    )

In [ ]:
import subprocess

comparison_summaries = []

# Run the original 2007 REleASE path once against the full CSV set.
ORIGINAL_RELEASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("=== Running original REleASE batch ===")
for csv_path in RELEASE_INPUT_PATHS:
    csv_path = Path(csv_path)
    try:
        rel_name = csv_path.relative_to(GSEP_ROOT).with_suffix("").as_posix().replace("/", "__")
    except ValueError:
        rel_name = csv_path.stem

    out_json_path = ORIGINAL_RELEASE_OUTPUT_DIR / f"release_{rel_name}.json"
    release_command = [
        "python",
        str(RELEASE_SCRIPT),
        "--model",
        "original-release",
        "--electron-csv",
        str(csv_path),
        "--electron-time-col",
        TIME_COL,
        "--electron-flux-col",
        ELECTRON_COL,
        "--proton-csv",
        str(csv_path),
        "--proton-time-col",
        TIME_COL,
        "--proton-gt10-col",
        PROTON_COL,
        "--release-matrix-json",
        str(RELEASE_MATRIX_JSON),
        "--forecast-lead-minutes",
        str(LEAD_MINUTES),
        "--paper-warning-trigger-pfu",
        "10",
        "--release-warning-trigger-pfu",
        "10",
        "--out-json",
        str(out_json_path),
    ]

    completed = subprocess.run(release_command, capture_output=True, text=True, check=False)
    print(f"[original] {csv_path.name} -> rc={completed.returncode}")
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)

original_compare_csv = ORIGINAL_RELEASE_OUTPUT_DIR / "release_hit_miss_comparison.csv"
original_compare_summary_json = ORIGINAL_RELEASE_OUTPUT_DIR / "release_hit_miss_summary.json"
original_compare_command = [
    "python",
    str(COMPARE_SCRIPT),
    "--gsep-list-csv",
    str(GSEP_LIST_CSV),
    "--release-output-dir",
    str(ORIGINAL_RELEASE_OUTPUT_DIR),
    "--out-csv",
    str(original_compare_csv),
    "--out-summary-json",
    str(original_compare_summary_json),
]
completed_original_compare = subprocess.run(original_compare_command, capture_output=True, text=True, check=False)
print(f"[original] compare rc={completed_original_compare.returncode}")
print(completed_original_compare.stdout)
if completed_original_compare.returncode != 0:
    print(completed_original_compare.stderr)

if original_compare_summary_json.exists():
    summary_payload = json.loads(original_compare_summary_json.read_text())
    comparison_summaries.append(
        {
            "label": "original-release",
            "summary_json": str(original_compare_summary_json),
            "counts": summary_payload.get("counts", {}),
            "rates": summary_payload.get("rates", {}),
        }
    )

# Run the HESPERIA three-matrix voter for each trained bin size.
for cfg in TRAINED_BIN_CONFIGS:
    bin_size = int(cfg["bin_size"])
    release_output_dir = Path(cfg["bin_dir"])
    matrix_library_json = Path(cfg["matrix_library_json"])
    release_output_dir.mkdir(parents=True, exist_ok=True)
    print(f"=== Running HESPERIA batch for bin={bin_size} ===")

    for csv_path in RELEASE_INPUT_PATHS:
        csv_path = Path(csv_path)
        try:
            rel_name = csv_path.relative_to(GSEP_ROOT).with_suffix("").as_posix().replace("/", "__")
        except ValueError:
            rel_name = csv_path.stem

        out_json_path = release_output_dir / f"release_{rel_name}.json"
        release_command = [
            "python",
            str(RELEASE_SCRIPT),
            "--model",
            "hesperia-release",
            "--instrument",
            RELEASE_INSTRUMENT,
            "--delta-matrix-library-json",
            str(matrix_library_json),
            "--ensemble-window-minutes",
            "30",
            "60",
            "90",
            "--electron-csv",
            str(csv_path),
            "--electron-time-col",
            TIME_COL,
            "--electron-flux-col",
            ELECTRON_COL,
            "--proton-csv",
            str(csv_path),
            "--proton-time-col",
            TIME_COL,
            "--proton-gt10-col",
            PROTON_COL,
            "--forecast-lead-minutes",
            str(LEAD_MINUTES),
            "--paper-hazard-pfu",
            "20",
            "--paper-warning-trigger-pfu",
            "10",
            "--out-json",
            str(out_json_path),
        ]

        completed = subprocess.run(release_command, capture_output=True, text=True, check=False)
        print(f"[bin={bin_size}] {csv_path.name} -> rc={completed.returncode}")
        if completed.returncode != 0:
            print(completed.stdout)
            print(completed.stderr)

    failed = [r for r in comparison_summaries if r.get("label") == f"bin={bin_size}" and False]
    compare_csv = release_output_dir / f"release_hit_miss_comparison_bin_{bin_size}.csv"
    compare_summary_json = release_output_dir / f"release_hit_miss_summary_bin_{bin_size}.json"
    compare_command = [
        "python",
        str(COMPARE_SCRIPT),
        "--gsep-list-csv",
        str(GSEP_LIST_CSV),
        "--release-output-dir",
        str(release_output_dir),
        "--out-csv",
        str(compare_csv),
        "--out-summary-json",
        str(compare_summary_json),
    ]
    completed_compare = subprocess.run(compare_command, capture_output=True, text=True, check=False)
    print(f"[bin={bin_size}] compare rc={completed_compare.returncode}")
    print(completed_compare.stdout)
    if completed_compare.returncode != 0:
        print(completed_compare.stderr)

    if compare_summary_json.exists():
        summary_payload = json.loads(compare_summary_json.read_text())
        comparison_summaries.append(
            {
                "label": f"bin={bin_size}",
                "summary_json": str(compare_summary_json),
                "counts": summary_payload.get("counts", {}),
                "rates": summary_payload.get("rates", {}),
            }
        )

print("=== Comparison summary ===")
for row in comparison_summaries:
    counts = row.get("counts", {})
    rates = row.get("rates", {})
    print(
        f"{row['label']} close={counts.get('close')} not_found={counts.get('not_found')} close_rate={rates.get('close_rate')}"
    )